#Mục tiêu
chúng ta sẽ áp dụng quy trình: Bronze (Raw) -> Silver (Cleaned) -> Gold (Processed).

##Cài đặt và Tải dữ liệu (Giai đoạn BRONZE - Dữ liệu gốc)

In [ ]:
!pip install kagglehub pyspark

import kagglehub
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, to_timestamp, monotonically_increasing_id

# 1. Khởi tạo Spark
spark = SparkSession.builder.appName("DataVersioning").getOrCreate()

# 2. Tải Dataset từ Kaggle (Amazon Reviews)
path = kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products")
files = [f for f in os.listdir(path) if f.endswith('.csv')]
raw_data_path = os.path.join(path, files[0])

# 3. Đọc dữ liệu gốc (Bronze Layer)
df_bronze = spark.read.csv(raw_data_path, header=True, inferSchema=True)

# Lưu version Bronze (Dữ liệu thô nguyên bản) dưới dạng Parquet để truy xuất nhanh
df_bronze.write.mode("overwrite").parquet("data_v1_bronze.parquet")
print("Đã lưu Version 1: Bronze (Original Data)")

Using Colab cache for faster access to the 'consumer-reviews-of-amazon-products' dataset.
Đã lưu Version 1: Bronze (Original Data)


##Làm sạch dữ liệu (Giai đoạn SILVER - Version 2)

In [ ]:
from pyspark.sql.functions import col, try_to_timestamp, lit, expr

# Đọc lại từ bản Bronze
df_v2 = spark.read.parquet("data_v1_bronze.parquet")

# Định nghĩa các cột có dấu chấm bằng dấu huyền
column_rating = "`reviews.rating`"
column_text = "`reviews.text`"
column_date = "`reviews.date`"

# --- 1. DATA CLEANING ---

# Bước A: Sử dụng expr và try_cast để ép kiểu Rating về Double an toàn
# Những giá trị 'TRUE', 'FALSE' hoặc rác sẽ tự động thành NULL
df_silver = df_v2.withColumn("rating_numeric", expr(f"try_cast({column_rating} AS DOUBLE)"))

# Bước B: Loại bỏ các bản ghi thiếu thông tin (xóa luôn những dòng có rating rác vừa thành NULL)
df_silver = df_silver.dropna(subset=["rating_numeric", column_text])

# Bước C: Loại bỏ dữ liệu trùng lặp theo ID
df_silver = df_silver.dropDuplicates(["id"])

# Bước D: Lọc bỏ rating không hợp lệ (1-5)
df_silver = df_silver.filter((col("rating_numeric") >= 1) & (col("rating_numeric") <= 5))

# Bước E: Xử lý ngày tháng an toàn
date_format = "yyyy-MM-dd'T'HH:mm:ss.SSSX"
df_silver = df_silver.withColumn(
    "review_date_converted",
    try_to_timestamp(col(column_date), lit(date_format))
)

# --- 2. CHUẨN BỊ OUTPUT CHO SILVER ---
# Thay thế cột cũ bằng cột số đã làm sạch để các bước sau không bị lỗi định dạng
df_silver = df_silver.withColumn("reviews_rating_clean", col("rating_numeric"))

# Chọn các cột cần thiết để lưu (Loại bỏ các cột trung gian hoặc cột lỗi cũ nếu muốn)
# Ở đây tôi giữ lại cấu trúc cũ nhưng dùng cột đã cast thành công
df_silver_final = df_silver.drop("rating_numeric")

# Lưu dữ liệu xuống version 2
df_silver_final.write.mode("overwrite").parquet("data_v2_silver.parquet")

print("Đã lưu Version 2: Silver (Cleaned Data) thành công với try_cast!")

Đã lưu Version 2: Silver (Cleaned Data) thành công với try_cast!


##Chuẩn hóa dữ liệu (Giai đoạn GOLD - Version 3)

In [ ]:
from pyspark.sql.functions import col, lower, regexp_replace, monotonically_increasing_id

# 1. Đọc lại từ bản Silver (Dữ liệu đã sạch ở Bước 2)
df_v3 = spark.read.parquet("data_v2_silver.parquet")

# 2. Định nghĩa tên cột có dấu chấm bằng dấu huyền để tránh lỗi Unresolved Column
column_text_raw = "`reviews.text`"
column_rating_clean = "`reviews.rating`" # Hoặc reviews_rating_clean tùy bước 2 bạn đặt tên

# --- DATA NORMALIZATION ---

# 1. Chuẩn hóa Text: Viết thường và loại bỏ ký tự đặc biệt
# Sử dụng dấu huyền trực tiếp trong col()
df_gold = df_v3.withColumn("review_content_final", lower(col(column_text_raw)))
df_gold = df_gold.withColumn("review_content_final", regexp_replace(col("review_content_final"), "[^a-z0-9\\s]", ""))

# 2. Min-Max Scaling cho cột Rating (Đưa về khoảng 0-1)
# Công thức: (x - min) / (max - min). Rating từ 1-5 nên sẽ là (x-1)/4
df_gold = df_gold.withColumn("rating_normalized", (col(column_rating_clean) - 1) / 4)

# 3. Thêm cột Index duy nhất cho mỗi dòng
df_gold = df_gold.withColumn("row_index", monotonically_increasing_id())

# --- LƯU VERSION GOLD ---
# Chọn lọc các cột quan trọng để bộ dataset gọn nhẹ, sẵn sàng cho các bước sau
df_final_output = df_gold.select(
    "row_index",
    "id",
    column_rating_clean,
    "rating_normalized",
    "review_content_final",
    "review_date_converted"
)

df_final_output.write.mode("overwrite").parquet("data_v3_gold_final.parquet")

print("Đã lưu Version 3: Gold (Normalized & Ready) thành công!")

Đã lưu Version 3: Gold (Normalized & Ready) thành công!


##Kết quả cuối cùng (GOLD LAYER)

In [ ]:
spark.read.parquet("data_v3_gold_final.parquet").show(10)

+---------+--------------------+--------------+-----------------+--------------------+---------------------+
|row_index|                  id|reviews.rating|rating_normalized|review_content_final|review_date_converted|
+---------+--------------------+--------------+-----------------+--------------------+---------------------+
|        0|AV-XeQLWuC1rwyj_gbP5|             5|              1.0|a lazy mans drean...|  2018-05-23 00:00:00|
|        1|AVpfIfGA1cnluZ0-emyp|             5|              1.0|i needed a second...|  2016-08-11 00:00:00|
|        2|AVpfpK8KLJeJML43BCuD|             5|              1.0|bought the echo f...|  2016-12-15 00:00:00|
|        3|AVpftoij1cnluZ0-p5n2|             5|              1.0|i needed somethin...|  2015-08-12 00:00:00|
|        4|AVpgdkC8ilAPnD_xsvyi|             1|              0.0|worst product eve...|  2017-02-13 00:00:00|
|        5|AVph0EeEilAPnD_x9myq|             5|              1.0|my daughter like ...|  2017-04-17 00:00:00|
|        6|AVphPmHu

#DATA CLEANING

##Kiểm tra dữ liệu thiếu (NULL, NaN)

In [ ]:
from pyspark.sql.functions import col, sum

df_bronze.select([
    sum(col(f"`{c}`").isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

+---+---------+-----------+----+-----+-----+----------+-----------------+---------+----+------------+------------------+------------+-----------------+----------------+-------------------+----------+------------------+--------------+------------------+------------+-------------+----------------+----------+
| id|dateAdded|dateUpdated|name|asins|brand|categories|primaryCategories|imageURLs|keys|manufacturer|manufacturerNumber|reviews.date|reviews.dateAdded|reviews.dateSeen|reviews.doRecommend|reviews.id|reviews.numHelpful|reviews.rating|reviews.sourceURLs|reviews.text|reviews.title|reviews.username|sourceURLs|
+---+---------+-----------+----+-----+-----+----------+-----------------+---------+----+------------+------------------+------------+-----------------+----------------+-------------------+----------+------------------+--------------+------------------+------------+-------------+----------------+----------+
|  0|        0|          0|   0|    0|    0|         0|                0|   

##Xử lý dữ liệu thiếu

In [ ]:
# Rename toàn bộ cột (loại bỏ dấu .)
df_bronze = df_bronze.toDF(*[
    c.replace(".", "_") for c in df_bronze.columns
])

from pyspark.sql.functions import expr

# Tạo rating_numeric
df_bronze = df_bronze.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)

# Drop NULL
df_clean = df_bronze.dropna(
    subset=["rating_numeric", "reviews_text"]
)

##Xoá dữ liệu trùng (duplicate)

In [ ]:
# Đếm số dòng trước
before = df_clean.count()
print("Before:", before)

Before: 3951


In [ ]:
# Xóa duplicate
df_clean = df_clean.dropDuplicates(["id"])

In [ ]:
# Đếm lại sau khi xoá
after = df_clean.count()
print("After:", after)
print("Removed:", before - after)

After: 19
Removed: 3932


##Loại bỏ dữ liệu sai

###Giá trị âm bất hợp lý

In [ ]:
from pyspark.sql.functions import col

df_clean = df_clean.filter(
    (col("rating_numeric") >= 1) & (col("rating_numeric") <= 5)
)

# Nếu có cột số khác (ví dụ numHelpful)
df_clean = df_clean.filter(
    col("reviews_numHelpful") >= 0
)

###Dữ liệu lỗi format

In [ ]:
# Rating dạng sai (TRUE/FALSE, string)
from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)

# Date lỗi format
from pyspark.sql.functions import try_to_timestamp, lit

df_clean = df_clean.withColumn(
    "review_date_clean",
    try_to_timestamp(col("reviews_date"), lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))
)

# Loại bỏ các dòng date lỗi
df_clean = df_clean.filter(
    col("review_date_clean").isNotNull()
)

## Chuẩn hoá kiểu dữ liệu

###Chuyển String → Number

In [ ]:
from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)


In [ ]:
# Kiểm tra kiểu dữ liệu
df_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- primaryCategories: string (nullable = true)
 |-- imageURLs: string (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- reviews_date: string (nullable = true)
 |-- reviews_dateAdded: string (nullable = true)
 |-- reviews_dateSeen: string (nullable = true)
 |-- reviews_doRecommend: string (nullable = true)
 |-- reviews_id: string (nullable = true)
 |-- reviews_numHelpful: string (nullable = true)
 |-- reviews_rating: string (nullable = true)
 |-- reviews_sourceURLs: string (nullable = true)
 |-- reviews_text: string (nullable = true)
 |-- reviews_title: string (nullable = true)
 |-- reviews_username: string 

###Chuyển Date → Datetime

In [ ]:
from pyspark.sql.functions import try_to_timestamp, col, lit

df_clean = df_clean.withColumn(
    "review_date_clean",
    try_to_timestamp(
        col("reviews_date"),
        lit("yyyy-MM-dd'T'HH:mm:ss.SSSX")
    )
)

# Loại bỏ dữ liệu lỗi format
df_clean = df_clean.filter(
    col("review_date_clean").isNotNull()
)

In [ ]:
# Kiểm tra lại dữ liệu
print("\n===== SAMPLE DATA =====")
df_clean.show(10)


===== SAMPLE DATA =====
+--------------------+-------------------+-------------------+--------------------+----------+------+--------------------+--------------------+--------------------+--------------------+------------+------------------+--------------------+-----------------+--------------------+-------------------+----------+------------------+--------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-------------------+
|                  id|          dateAdded|        dateUpdated|                name|     asins| brand|          categories|   primaryCategories|           imageURLs|                keys|manufacturer|manufacturerNumber|        reviews_date|reviews_dateAdded|    reviews_dateSeen|reviews_doRecommend|reviews_id|reviews_numHelpful|reviews_rating|  reviews_sourceURLs|        reviews_text|       reviews_title|reviews_username|          sourceURLs|rating_numeric|  review_date_clean|
+----------------